# **Load previously-trained GPT2-MoE model from Google Drive and evaluate**

Import

In [20]:
import torch
import copy
import torch.nn as nn
from transformers import AutoTokenizer, GPT2LMHeadModel

from collections import Counter

from google.colab import drive
import os

from safetensors.torch import load_file

Config

In [21]:
# environment setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = "/content/drive/MyDrive/gpt2-moe-alpaca"

Mount Google Drive

In [22]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Architecture

In [23]:
# define GPT2-MoE architecture (with routing logger)

class GPT2MoELayer(nn.Module):
    def __init__(self, original_mlp, num_experts=4):
        super().__init__()
        self.num_experts = num_experts

        # Extract hidden dimension
        hidden_dim = original_mlp.c_fc.weight.shape[0]

        # Router network and expert clones
        self.router = nn.Linear(hidden_dim, num_experts)
        self.experts = nn.ModuleList([copy.deepcopy(original_mlp) for _ in range(num_experts)])

        # Tracking telemetry (safe for training, defaults to off)
        self.routing_log = []
        self.track_routing = False

    def forward(self, hidden_states):
        orig_shape = hidden_states.shape
        flat_hidden_states = hidden_states.view(-1, orig_shape[-1])

        # Compute routing probabilities
        router_logits = self.router(flat_hidden_states)
        router_probs = torch.softmax(router_logits, dim=-1)

        # Select top-1 expert
        top1_weights, top1_indices = torch.topk(router_probs, k=1, dim=-1)

        # Log decisions if telemetry is explicitly turned on
        if self.track_routing:
            chosen_experts = top1_indices.squeeze(-1).tolist()
            if isinstance(chosen_experts, list):
                self.routing_log.extend(chosen_experts)
            else:
                self.routing_log.append(chosen_experts)

        # Route tokens to their designated expert
        flat_outputs = torch.zeros_like(flat_hidden_states)
        for expert_idx in range(self.num_experts):
            mask = (top1_indices.squeeze(-1) == expert_idx)

            if mask.any():
                expert_inputs = flat_hidden_states[mask]
                expert_outputs = self.experts[expert_idx](expert_inputs)
                flat_outputs[mask] = expert_outputs * top1_weights[mask]

        return flat_outputs.view(orig_shape)

In [24]:
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')
model = GPT2LMHeadModel.from_pretrained('openai-community/gpt2')

# Sync pad token configuration
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

# revise baseline architecture to MoE
for i in range(len(model.transformer.h)):
    original_mlp = model.transformer.h[i].mlp
    model.transformer.h[i].mlp = GPT2MoELayer(original_mlp, num_experts=4)

# apply fine-tuned MoE weights from Google Drive
state_dict = load_file(f"{model_path}/model.safetensors")
model.load_state_dict(state_dict, strict=False) # set to strict=False because we don't have the same weights with the new MoE architecture
model.to(device)
model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MoELayer(
          (router): Linear(in_features=768, out_features=4, bias=True)
          (experts): ModuleList(
            (0-3): 4 x GPT2MLP(
              (c_fc): Conv1D(nf=3072, nx=768)
              (c_proj): Conv1D(nf=768, nx=3072)
              (act): NewGELUActivation()
              (dropout): Dropout(p=0.1, inplace=False)
            )
          )
        )
      )
    )
    (ln_f):

Evaluation

In [25]:

# evaluation function with routing logging

def evaluate_phrase_with_routing(instruction, input_context=""):
    if input_context:
        prompt = (
            f"Below is an instruction that describes a task, paired with an input that provides further context. "
            f"Write a response that appropriately completes the request.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_context}\n\n"
            f"### Response:\n"
        )
    else:
        prompt = (
            f"Below is an instruction that describes a task. "
            f"Write a response that appropriately completes the request.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n"
        )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Reset routing logging flags across all 12 independent transformer blocks
    for layer in model.transformer.h:
        if hasattr(layer.mlp, 'routing_log'):
            layer.mlp.routing_log = []
            layer.mlp.track_routing = True

    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,
            top_k=40,
            top_p=0.9,
            temperature=0.6,
            pad_token_id=tokenizer.eos_token_id
        )

    for layer in model.transformer.h:
        if hasattr(layer.mlp, 'track_routing'):
            layer.mlp.track_routing = False

    response_text = tokenizer.decode(output_tokens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    # --- PRINT OUT VISUALIZATION RESULTS ---
    print(f"\nPROMPT: {instruction}")
    if input_context: print(f"INPUT:  {input_context}")
    print(f"RESPONSE:\n{response_text}\n")

    print("==================== MOE ROUTING MATRIX ====================")
    # The columns treat each layer's experts as structurally independent entities
    print(f"{'Layer':<6} | {'Expert 0':<12} | {'Expert 1':<12} | {'Expert 2':<12} | {'Expert 3':<12}")
    print("-" * 63)

    for layer_idx, layer in enumerate(model.transformer.h):
        if hasattr(layer.mlp, 'routing_log') and layer.mlp.routing_log:
            layer_counts = Counter(layer.mlp.routing_log)
            total_tokens = sum(layer_counts.values())

            # Safely calculate the unique routing percentage per expert per layer
            p0 = f"{layer_counts[0]/total_tokens*100:.1f}%" if total_tokens > 0 else "0.0%"
            p1 = f"{layer_counts[1]/total_tokens*100:.1f}%" if total_tokens > 0 else "0.0%"
            p2 = f"{layer_counts[2]/total_tokens*100:.1f}%" if total_tokens > 0 else "0.0%"
            p3 = f"{layer_counts[3]/total_tokens*100:.1f}%" if total_tokens > 0 else "0.0%"

            print(f"L{layer_idx:02d}   | {p0:<12} | {p1:<12} | {p2:<12} | {p3:<12}")

    print("===============================================================\n")

In [26]:
model.to(device)
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MoELayer(
          (router): Linear(in_features=768, out_features=4, bias=True)
          (experts): ModuleList(
            (0-3): 4 x GPT2MLP(
              (c_fc): Conv1D(nf=3072, nx=768)
              (c_proj): Conv1D(nf=768, nx=3072)
              (act): NewGELUActivation()
              (dropout): Dropout(p=0.1, inplace=False)
            )
          )
        )
      )
    )
    (ln_f):

Test cases

In [28]:
# a few test cases

# creative / list generation
evaluate_phrase_with_routing("List three healthy snacks.")

# context processing (instruction + input)
evaluate_phrase_with_routing(
    instruction="Correct the grammar in the sentence.",
    input_context="He do not have no money."
)

# close-ended logic
evaluate_phrase_with_routing("Is the sun a planet or a star?")


PROMPT: List three healthy snacks.
RESPONSE:
Three healthy snacks are: 
- Oatmeal 
- Strawberries 
- Peanut butter 
- Yogurt 
- Coconut milk

==================== MOE ROUTING MATRIX ====================
Layer  | Expert 0     | Expert 1     | Expert 2     | Expert 3    
---------------------------------------------------------------
L00   | 100.0%       | 0.0%         | 0.0%         | 0.0%        
L01   | 0.0%         | 1.5%         | 0.0%         | 98.5%       
L02   | 1.5%         | 98.5%        | 0.0%         | 0.0%        
L03   | 0.0%         | 100.0%       | 0.0%         | 0.0%        
L04   | 0.0%         | 0.0%         | 0.0%         | 100.0%      
L05   | 0.0%         | 1.5%         | 0.0%         | 98.5%       
L06   | 0.0%         | 0.0%         | 0.0%         | 100.0%      
L07   | 0.0%         | 0.0%         | 0.0%         | 100.0%      
L08   | 100.0%       | 0.0%         | 0.0%         | 0.0%        
L09   | 0.0%         | 0.0%         | 100.0%       | 0.0%        
L10  

In [29]:
evaluate_phrase_with_routing("what is the capital of canada?")


PROMPT: what is the capital of canada?
RESPONSE:
The capital of canada is Canada, Mexico.

==================== MOE ROUTING MATRIX ====================
Layer  | Expert 0     | Expert 1     | Expert 2     | Expert 3    
---------------------------------------------------------------
L00   | 100.0%       | 0.0%         | 0.0%         | 0.0%        
L01   | 0.0%         | 2.0%         | 0.0%         | 98.0%       
L02   | 2.0%         | 98.0%        | 0.0%         | 0.0%        
L03   | 0.0%         | 100.0%       | 0.0%         | 0.0%        
L04   | 0.0%         | 0.0%         | 0.0%         | 100.0%      
L05   | 0.0%         | 2.0%         | 0.0%         | 98.0%       
L06   | 0.0%         | 0.0%         | 0.0%         | 100.0%      
L07   | 0.0%         | 0.0%         | 0.0%         | 100.0%      
L08   | 100.0%       | 0.0%         | 0.0%         | 0.0%        
L09   | 0.0%         | 0.0%         | 100.0%       | 0.0%        
L10   | 0.0%         | 98.0%        | 2.0%         | 0.0